In [5]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# =========================================================
# 1. LOAD FILES
# =========================================================
train = pd.read_csv("GDP_Forecast_train.csv")
test = pd.read_csv("GDP_Forecast_test.csv")
gdp_pca = pd.read_csv("GDP_PCA.csv")
unrate = pd.read_csv("UNRATE (1).csv")
comp = pd.read_csv("compustat_quarterly_financials.csv")

# =========================================================
# 2. DATE HELPERS
# =========================================================
def yq_to_date(yq):
    year = int(yq[:4])
    quarter = int(yq[-1])
    month_map = {1: 1, 2: 4, 3: 7, 4: 10}
    return pd.Timestamp(year=year, month=month_map[quarter], day=1)

def date_to_yq(dt):
    q = (dt.month - 1) // 3 + 1
    return f"{dt.year}Q{q}"

# training / test dates
train["date"] = train["YQ"].apply(yq_to_date)
test["date"] = test["YQ"].apply(yq_to_date)

# =========================================================
# 3. PREPARE GDP_PCA (VALIDATION ONLY)
# =========================================================
# observation_date format looks like 1/1/1990, 1/4/1990, etc.
gdp_pca["date"] = pd.to_datetime(gdp_pca["observation_date"], dayfirst=True)
gdp_pca["YQ"] = gdp_pca["date"].apply(date_to_yq)
gdp_pca = gdp_pca[["YQ", "GDP_PCA"]].copy()

# =========================================================
# 4. PREPARE UNRATE -> QUARTERLY
# =========================================================
unrate["date"] = pd.to_datetime(unrate["observation_date"])
unrate["year"] = unrate["date"].dt.year
unrate["quarter"] = unrate["date"].dt.quarter

unrate_q = (
    unrate.groupby(["year", "quarter"], as_index=False)["UNRATE"]
    .mean()
)

unrate_q["YQ"] = unrate_q.apply(
    lambda r: f"{int(r['year'])}Q{int(r['quarter'])}", axis=1
)
unrate_q["date"] = unrate_q["YQ"].apply(yq_to_date)
unrate_q = unrate_q[["YQ", "date", "UNRATE"]].copy()

# =========================================================
# 5. PREPARE COMPUSTAT QUARTERLY AGGREGATES
# =========================================================
# compustat file already has quarterly field datacqtr like 1990Q1
# use cross-sectional mean and median for robustness
comp["YQ"] = comp["datacqtr"].astype(str)

comp_q_mean = (
    comp.groupby("YQ", as_index=False)[["atq", "ltq", "niq"]]
    .mean()
    .rename(columns={
        "atq": "atq_mean",
        "ltq": "ltq_mean",
        "niq": "niq_mean"
    })
)

comp_q_median = (
    comp.groupby("YQ", as_index=False)[["atq", "ltq", "niq"]]
    .median()
    .rename(columns={
        "atq": "atq_med",
        "ltq": "ltq_med",
        "niq": "niq_med"
    })
)

comp_q = comp_q_mean.merge(comp_q_median, on="YQ", how="outer")

# =========================================================
# 6. BUILD MASTER PANEL
# =========================================================
full = pd.concat([
    train[["YQ", "date", "NGDP"]],
    test[["YQ", "date", "NGDP"]]
], ignore_index=True)

full = full.merge(unrate_q[["YQ", "UNRATE"]], on="YQ", how="left")
full = full.merge(comp_q, on="YQ", how="left")
full = full.merge(gdp_pca, on="YQ", how="left")

full["is_train"] = full["YQ"].isin(train["YQ"]).astype(int)
full["is_test"] = full["YQ"].isin(test["YQ"]).astype(int)

# IMPORTANT: remove placeholder test NGDP so recursive prediction can fill it
full.loc[full["is_test"] == 1, "NGDP"] = np.nan

# =========================================================
# 7. FEATURE ENGINEERING
# =========================================================
def safe_pct_change(series):
    return series.pct_change().replace([np.inf, -np.inf], np.nan)

def make_features(df):
    df = df.copy()
    df = df.sort_values("date").reset_index(drop=True)

    # ---- lagged NGDP ----
    df["NGDP_lag1"] = df["NGDP"].shift(1)
    df["NGDP_lag2"] = df["NGDP"].shift(2)
    df["NGDP_lag4"] = df["NGDP"].shift(4)

    # ---- unemployment features ----
    df["UNRATE_lag1"] = df["UNRATE"].shift(1)
    df["UNRATE_lag2"] = df["UNRATE"].shift(2)
    df["UNRATE_change"] = df["UNRATE"] - df["UNRATE"].shift(1)
    df["UNRATE_change_lag1"] = df["UNRATE"].shift(1) - df["UNRATE"].shift(2)
    df["UNRATE_accel"] = df["UNRATE_change"] - df["UNRATE_change_lag1"]
    df["UNRATE_spike"] = np.maximum(df["UNRATE_change"], 0)
    df["UNRATE_spike2"] = df["UNRATE_spike"] ** 2

    # ---- financial aggregate levels ----
    for col in ["atq_mean", "ltq_mean", "niq_mean", "atq_med", "ltq_med", "niq_med"]:
        df[f"{col}_lag1"] = df[col].shift(1)
        df[f"{col}_lag2"] = df[col].shift(2)
        df[f"{col}_growth"] = safe_pct_change(df[col])
        df[f"{col}_growth_lag1"] = df[f"{col}_growth"].shift(1)

    # ---- financial stress proxies ----
    df["leverage_mean"] = df["ltq_mean"] / df["atq_mean"]
    df["profitability_mean"] = df["niq_mean"] / df["atq_mean"]
    df["leverage_med"] = df["ltq_med"] / df["atq_med"]
    df["profitability_med"] = df["niq_med"] / df["atq_med"]

    df["leverage_mean_lag1"] = df["leverage_mean"].shift(1)
    df["profitability_mean_lag1"] = df["profitability_mean"].shift(1)
    df["leverage_med_lag1"] = df["leverage_med"].shift(1)
    df["profitability_med_lag1"] = df["profitability_med"].shift(1)

    # ---- time features ----
    df["year"] = df["date"].dt.year
    df["quarter"] = df["date"].dt.quarter

    # quarter dummies (optional encoded numerically for tree models)
    df["q1"] = (df["quarter"] == 1).astype(int)
    df["q2"] = (df["quarter"] == 2).astype(int)
    df["q3"] = (df["quarter"] == 3).astype(int)
    df["q4"] = (df["quarter"] == 4).astype(int)

    return df

full = make_features(full)

# =========================================================
# 8. DEFINE STRESS REGIME
# =========================================================
# define thresholds USING TRAINING PERIOD ONLY
tmp_train = full[full["is_train"] == 1].copy()

# use positive unemployment spikes and high unemployment as regime signals
spike_thr = tmp_train["UNRATE_change"].quantile(0.90)
level_thr = tmp_train["UNRATE"].quantile(0.80)

# fallback if quantile becomes NaN
if pd.isna(spike_thr):
    spike_thr = 0.4
if pd.isna(level_thr):
    level_thr = 6.5

def add_regime_flags(df, spike_thr, level_thr):
    df = df.copy()

    # stress if unemployment is high OR unemployment jumps materially
    df["stress_flag"] = (
        (df["UNRATE_change"] >= spike_thr) |
        (df["UNRATE"] >= level_thr) |
        (df["NGDP_lag1"] < 0)
    ).astype(int)

    # interaction terms
    df["stress_x_unrate_change"] = df["stress_flag"] * df["UNRATE_change"]
    df["stress_x_unrate"] = df["stress_flag"] * df["UNRATE"]
    df["stress_x_ngdp_lag1"] = df["stress_flag"] * df["NGDP_lag1"]

    return df

full = add_regime_flags(full, spike_thr, level_thr)

# =========================================================
# 9. SELECT FEATURES
# =========================================================
feature_cols = [
    # NGDP dynamics
    "NGDP_lag1", "NGDP_lag2", "NGDP_lag4",

    # unemployment / macro stress
    "UNRATE", "UNRATE_lag1", "UNRATE_lag2",
    "UNRATE_change", "UNRATE_change_lag1", "UNRATE_accel",
    "UNRATE_spike", "UNRATE_spike2",

    # compustat aggregates
    "atq_mean_lag1", "ltq_mean_lag1", "niq_mean_lag1",
    "atq_med_lag1", "ltq_med_lag1", "niq_med_lag1",

    "atq_mean_growth_lag1", "ltq_mean_growth_lag1", "niq_mean_growth_lag1",
    "atq_med_growth_lag1", "ltq_med_growth_lag1", "niq_med_growth_lag1",

    "leverage_mean_lag1", "profitability_mean_lag1",
    "leverage_med_lag1", "profitability_med_lag1",

    # regime features
    "stress_flag", "stress_x_unrate_change", "stress_x_unrate", "stress_x_ngdp_lag1",

    # quarter
    "q1", "q2", "q3", "q4"
]

# =========================================================
# 10. TRAIN DATA
# =========================================================
train_model = full[full["is_train"] == 1].copy()
train_model = train_model.dropna(subset=feature_cols + ["NGDP"]).reset_index(drop=True)

X_train = train_model[feature_cols]
y_train = train_model["NGDP"]

# split by regime
train_normal = train_model[train_model["stress_flag"] == 0].copy()
train_stress = train_model[train_model["stress_flag"] == 1].copy()

X_train_normal = train_normal[feature_cols]
y_train_normal = train_normal["NGDP"]

X_train_stress = train_stress[feature_cols]
y_train_stress = train_stress["NGDP"]

print("Training rows total:", len(train_model))
print("Normal regime rows:", len(train_normal))
print("Stress regime rows:", len(train_stress))
print("Stress threshold (UNRATE_change):", round(spike_thr, 4))
print("High unemployment threshold:", round(level_thr, 4))

# =========================================================
# 11. FIT MODELS
# =========================================================
# fallback linear model on full sample
ridge_all = Ridge(alpha=2.0)
ridge_all.fit(X_train, y_train)

# random forest on full sample (stability)
rf_all = RandomForestRegressor(
    n_estimators=400,
    max_depth=5,
    min_samples_leaf=2,
    random_state=42
)
rf_all.fit(X_train, y_train)

# normal-regime model
xgb_normal = XGBRegressor(
    n_estimators=400,
    max_depth=3,
    learning_rate=0.04,
    subsample=0.90,
    colsample_bytree=0.90,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42
)
xgb_normal.fit(X_train_normal, y_train_normal)

# stress-regime model
# if too few stress rows, fit on full sample as fallback
if len(train_stress) >= 12:
    xgb_stress = XGBRegressor(
        n_estimators=500,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.95,
        colsample_bytree=0.95,
        reg_alpha=0.0,
        reg_lambda=1.0,
        objective="reg:squarederror",
        random_state=42
    )
    xgb_stress.fit(X_train_stress, y_train_stress)
    stress_model_is_real = True
else:
    xgb_stress = XGBRegressor(
        n_estimators=450,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.90,
        colsample_bytree=0.90,
        reg_alpha=0.0,
        reg_lambda=1.0,
        objective="reg:squarederror",
        random_state=42
    )
    xgb_stress.fit(X_train, y_train)
    stress_model_is_real = False

print("Stress-specific model trained on stress sample only:", stress_model_is_real)

# =========================================================
# 12. RECURSIVE FORECAST ON TEST SET
# =========================================================
full_pred = full.copy()
pred_records = []

test_idx = full_pred.index[full_pred["is_test"] == 1].tolist()

for idx in test_idx:
    # rebuild features each iteration because lagged NGDP must use prior predictions
    full_pred = make_features(full_pred)
    full_pred = add_regime_flags(full_pred, spike_thr, level_thr)

    row = full_pred.loc[idx, feature_cols]

    if row.isna().any():
        print(f"Skipping {full_pred.loc[idx, 'YQ']} due to missing predictors.")
        continue

    x_row = pd.DataFrame([row.values], columns=feature_cols)

    p_ridge = ridge_all.predict(x_row)[0]
    p_rf = rf_all.predict(x_row)[0]

    # choose regime-specific xgb
    stress_now = int(full_pred.loc[idx, "stress_flag"])

    if stress_now == 1:
        p_xgb = xgb_stress.predict(x_row)[0]

        # stronger reaction in stress periods:
        # more weight on stress model, some ridge/rf for stabilization
        pred = 0.20 * p_ridge + 0.15 * p_rf + 0.65 * p_xgb

        # optional shock amplification based on unemployment spike
        # keeps the model more responsive to crisis-like quarters
        shock_adj = 0

        if full_pred.loc[idx, "UNRATE_change"] > spike_thr:
            shock_adj = -1.2 * full_pred.loc[idx, "UNRATE_spike"]

        pred = pred + shock_adj

        # prevent unrealistic extreme forecasts
        pred = np.clip(pred, -30, 30)

    else:
        p_xgb = xgb_normal.predict(x_row)[0]
        pred = 0.20 * p_ridge + 0.20 * p_rf + 0.60 * p_xgb

    pred_records.append({
        "YQ": full_pred.loc[idx, "YQ"],
        "stress_flag": stress_now,
        "p_ridge": p_ridge,
        "p_rf": p_rf,
        "p_xgb": p_xgb,
        "NGDP_pred": pred
    })

    # recursive update
    full_pred.loc[idx, "NGDP"] = pred

pred_df = pd.DataFrame(pred_records)

# =========================================================
# 13. VALIDATE AGAINST GDP_PCA PROXY
# =========================================================
pred_df = pred_df.merge(gdp_pca[["YQ", "GDP_PCA"]], on="YQ", how="left")

valid = pred_df.dropna(subset=["GDP_PCA"]).copy()

rmse = np.sqrt(mean_squared_error(valid["GDP_PCA"], valid["NGDP_pred"]))
mae = np.mean(np.abs(valid["GDP_PCA"] - valid["NGDP_pred"]))

print("\nValidation against GDP_PCA proxy")
print("RMSE:", round(rmse, 6))
print("MAE :", round(mae, 6))

print("\nPredictions vs GDP_PCA:")
print(valid[["YQ", "stress_flag", "NGDP_pred", "GDP_PCA"]])

# =========================================================
# 14. OPTIONAL: NORMAL-PERIOD VS CRISIS-PERIOD DIAGNOSTICS
# =========================================================
pre2020 = valid[valid["YQ"] < "2020Q1"].copy()
yr2020 = valid[valid["YQ"] >= "2020Q1"].copy()

if len(pre2020) > 0:
    rmse_pre2020 = np.sqrt(mean_squared_error(pre2020["GDP_PCA"], pre2020["NGDP_pred"]))
    print("\nRMSE for 2016Q1-2019Q4:", round(rmse_pre2020, 6))

if len(yr2020) > 0:
    rmse_2020 = np.sqrt(mean_squared_error(yr2020["GDP_PCA"], yr2020["NGDP_pred"]))
    print("RMSE for 2020Q1-2020Q4:", round(rmse_2020, 6))

# =========================================================
# 15. SAVE SUBMISSION
# =========================================================
submission = test[["YQ"]].merge(pred_df[["YQ", "NGDP_pred"]], on="YQ", how="left")
submission = submission.rename(columns={"NGDP_pred": "NGDP"})
submission["NGDP"] = submission["NGDP"].round(1)

submission.to_csv("NGDP_submission_regime_aware.csv", index=False)

print("\nSaved: NGDP_submission_regime_aware.csv")
print(submission)

Training rows total: 100
Normal regime rows: 71
Stress regime rows: 29
Stress threshold (UNRATE_change): 0.3333
High unemployment threshold: 7.3667
Stress-specific model trained on stress sample only: True

Validation against GDP_PCA proxy
RMSE: 7.862794
MAE : 3.093302

Predictions vs GDP_PCA:
        YQ  stress_flag  NGDP_pred  GDP_PCA
0   2016Q1            0   4.204906      2.0
1   2016Q2            0   4.138075      4.1
2   2016Q3            0   3.149537      3.9
3   2016Q4            0   3.287281      4.2
4   2017Q1            0   3.755242      4.1
5   2017Q2            0   4.305801      3.3
6   2017Q3            0   2.382341      5.3
7   2017Q4            0   3.937854      7.2
8   2018Q1            0   4.861630      5.9
9   2018Q2            0   4.230366      5.1
10  2018Q3            0   4.116193      4.3
11  2018Q4            0   3.697015      2.3
12  2019Q1            0   2.790054      3.8
13  2019Q2            0   4.183546      5.5
14  2019Q3            0   4.110442      6.1
1